# Calorie Lens: YOLO11n-seg training (Colab, free T4)

Runtime: **Runtime > Change runtime type > T4 GPU**. Run the cells top to bottom.
Outputs go to Google Drive so a disconnect does not lose them: `best.pt`, `detector.onnx`, `detector.labels.json`.

Class order is defined by `ml/labels.yaml` (19 classes: 15 dishes + 4 containers). Never reorder it.

In [ ]:
# 1. Mount Drive and get the repo's ml/ folder.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/calorie-lens'   # create this folder in Drive
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Option A (default): copy the ml/ folder you uploaded to Drive:  MyDrive/calorie-lens/ml/
# Option B: set REPO_URL to your git repo and it will be cloned instead.
REPO_URL = ''          # e.g. 'https://github.com/<you>/calorie-lens.git'
ML_DIR = '/content/ml'
shutil.rmtree(ML_DIR, ignore_errors=True)
if REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/repo'], check=True)
    shutil.copytree('/content/repo/ml', ML_DIR)
else:
    shutil.copytree(f'{DRIVE_ROOT}/ml', ML_DIR)
print(os.listdir(ML_DIR))

In [ ]:
# 2. Install. ultralytics pinned to a version verified to exist on PyPI (8.4.193).
!pip -q install ultralytics==8.4.193 onnx onnxslim onnxruntime pyyaml
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Unpack datasets. Put Roboflow "YOLOv8/YOLO11" (segmentation) export zips in
#    MyDrive/calorie-lens/exports/*.zip  and your own labelled meals (files named <meal-id>__<n>.jpg)
#    in MyDrive/calorie-lens/mine.zip.  Each zip becomes /content/exports/<zipname>/.
import glob, zipfile
os.makedirs('/content/exports', exist_ok=True)
for z in sorted(glob.glob(f'{DRIVE_ROOT}/exports/*.zip') + glob.glob(f'{DRIVE_ROOT}/mine.zip')):
    name = os.path.splitext(os.path.basename(z))[0]
    dest = f'/content/exports/{name}'
    if not os.path.isdir(dest):
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
    print(name, '->', dest)

In [ ]:
# 4. Build the merged dataset. Edit SOURCES: 'dir' for data already in our class order,
#    'dir::ml/mappings/<name>.yaml' for seed sets that need remapping.
SOURCES = [
    '/content/exports/mine',
    # '/content/exports/tw_food::/content/ml/mappings/tw_food.yaml',
]
args = ' '.join(f'--source "{s}"' for s in SOURCES)
!python {ML_DIR}/make_dataset_yaml.py --out /content/ds {args} --dry-run
# Check the per-class table above: every class should have data in train AND val. Then build for real:
!python {ML_DIR}/make_dataset_yaml.py --out /content/ds {args}
!head -30 /content/ds/data.yaml

In [ ]:
# 5. Train. Defaults kept on purpose: HSV augmentation (hsv_h/s/v) untouched; mosaic on; horizontal flip on.
from ultralytics import YOLO
model = YOLO('yolo11n-seg.pt')
results = model.train(
    data='/content/ds/data.yaml',
    imgsz=640,
    epochs=100,
    patience=20,
    batch=16,
    mosaic=1.0,
    fliplr=0.5,
    project=f'{DRIVE_ROOT}/runs',   # on Drive so a disconnect can be resumed
    name='calorie_seg',
    exist_ok=True,
    seed=0,
    workers=2,
)
# If Colab disconnects: re-run cells 1-4, then
#   YOLO(f'{DRIVE_ROOT}/runs/calorie_seg/weights/last.pt').train(resume=True)

In [ ]:
# 6. Validate best.pt (box + mask mAP50, per class).
BEST = f'{DRIVE_ROOT}/runs/calorie_seg/weights/best.pt'
best = YOLO(BEST)
m = best.val(data='/content/ds/data.yaml', imgsz=640)
print('box mAP50 %.3f | mask mAP50 %.3f' % (m.box.map50, m.seg.map50))
for k, ci in enumerate(m.seg.ap_class_index):
    print(f'{best.names[int(ci)]:<22} mask AP50 {m.seg.ap50[k]:.3f}')

In [ ]:
# 7. Keep a dated copy of best.pt on Drive (the macOS Core ML workflow takes this file).
import datetime
stamp = datetime.date.today().isoformat()
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
shutil.copy(BEST, f'{DRIVE_ROOT}/models/best_{stamp}.pt')
shutil.copy(BEST, f'{DRIVE_ROOT}/models/best.pt')
print(os.listdir(f'{DRIVE_ROOT}/models'))

In [ ]:
# 8. Export ONNX for the web: imgsz 320, opset 17, simplify.
#    detector.onnx       = DETECT head [1, 4+19, 2100]  (what web/src/detector.ts decodes)  + detector.labels.json
#    detector_seg.onnx   = SEG head [1, 4+19+32, 2100] + proto [1, 32, 80, 80] (for a future mask decoder)
!python {ML_DIR}/export_onnx.py {BEST} --out /content/export --also-seg
os.makedirs(f'{DRIVE_ROOT}/export', exist_ok=True)
for f in os.listdir('/content/export'):
    shutil.copy(f'/content/export/{f}', f'{DRIVE_ROOT}/export/{f}')
!ls -la /content/export

In [ ]:
# 9. (Optional) Evaluate on a HELD-OUT set of your own meals (never in the merge above).
#    Upload heldout.zip (YOLO-seg export or images/ + labels/) to MyDrive/calorie-lens/.
HELD = f'{DRIVE_ROOT}/heldout.zip'
if os.path.exists(HELD):
    with zipfile.ZipFile(HELD) as f:
        f.extractall('/content/heldout')
    !python {ML_DIR}/eval.py --model {BEST} --data /content/heldout --out {DRIVE_ROOT}/eval_out
    from IPython.display import Image, display
    display(Image(f'{DRIVE_ROOT}/eval_out/top1_confusion.png'))
else:
    print('no heldout.zip yet; skipping')

## Next steps
1. Download `export/detector.onnx` and `export/detector.labels.json` from Drive into `web/public/models/`.
2. Run the GitHub Actions workflow `.github/workflows/coreml-export.yml` with `best.pt` (macOS runner) for Core ML.
3. Exit criterion (PLAN M3): at least 85% top-1 on held-out meals.